# 🛠️ Data Engineering Assistant (RAG Baseline)

Building a dense vector retrieval application using **LangChain**, **Gemini**, and **Qdrant**.

### The Core RAG Loop
1. **Load** the Big Book of Data Engineering PDF.
2. **Split** it into smaller, overlapping chunks.
3. **Embed** those chunks into a mathematical space.
4. **Store** the embeddings in a Qdrant vector database.
5. **Retrieve** relevant chunks based on a user's question.
6. **Generate** an answer grounded *strictly* in the retrieved context.

### 🧠 Mental Model: The Two AI Brains
It is a common beginner pitfall to confuse the two AI models used in this pipeline. 
* **The Embedding Model (`gemini-embedding-001`):** Think of this as the **Librarian**. It doesn't speak or answer questions. Its only job is to read text and assign it a mathematical coordinate (a vector) so it can be filed on a shelf. 
* **The Generation Model (`gemini-2.5-flash`):** Think of this as the **Analyst**. It doesn't know how to find the books. It relies entirely on the Librarian to hand it the right pages, and then it reads those pages to write a coherent summary for the user.

In [1]:
import warnings
warnings.filterwarnings("ignore")

| Component | Framework / Tool | What It Does In Your Notebook |
| :--- | :--- | :--- |
| **Environment Manager** | `uv` | A fast Rust-based Python package and virtual environment manager replacing pip and venv. |
| **Orchestrator** | `LangChain` | Provides abstraction layers (Document, VectorStore, Prompts) and chains them together using LCEL (`prompt | llm | parser`). |
| **Embedding Engine** | `models/gemini-embedding-001` | Converts text chunks into dense numeric vectors where semantic meaning is captured geometrically. |
| **Vector Database** | `Qdrant` | An in-memory vector database (`location=":memory:"`) storing vectors alongside metadata payloads for similarity lookup. |
| **Generator LLM** | `gemini-2.5-flash` | The core intelligence model that synthesizes retrieved context into human answers. |
| **Document Reader** | `pypdf` | Extracts raw text streams and page counts from local PDF files. |

### 📦 Environmental Setup

We use `uv`, a hyper-fast Python package manager written in Rust, to handle our environment. 

**Terminal Commands to run before starting:**
```bash
# 1. Install uv (macOS/Linux)
curl -LsSf [https://astral.sh/uv/install.sh](https://astral.sh/uv/install.sh) | sh

# 2. Initialize project and activate virtual environment
uv init
uv venv
source .venv/bin/activate  # On Windows: .venv\Scripts\activate

# 3. Create requirements.txt and install
echo "langchain-core langchain-community langchain-text-splitters langchain-google-genai langchain-qdrant qdrant-client python-dotenv ipykernel jupyter pypdf tenacity" > requirements.txt
uv pip install -r requirements.txt

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_google_genai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [2]:
import os
import logging
import textwrap
from pathlib import Path
from typing import List, Union, Optional, Dict
from dotenv import load_dotenv

from langchain_core.vectorstores import VectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tenacity import retry, wait_fixed, stop_after_attempt

# Load environment variables (.env file containing GEMINI_API_KEY)
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

An embedding model turns a piece of text into a numeric vector. If two pieces of text are related, their vectors should be close to each other. We measure that closeness with cosine similarity, which tells us how aligned two vectors are.

Why this matters
In RAG, we embed each document chunk once, then embed the user’s question and search for the most similar chunks. That retrieval step is what helps the model answer from the right part of the PDF.

### 📄 1. Document Loading & Memory Optimization

We use `PyPDFLoader` to extract text from our target document. 

**Pro-Tip: Lazy Loading**
Standard loading reads an entire file into RAM at once. By using `loader.lazy_load()`, we yield pages one by one as an iterator. This keeps the memory footprint flat, allowing you to parse gigabyte-scale documents effortlessly on a standard laptop.

In [8]:
def load_pdf_documents(
    file_path: Union[str, Path], 
    extra_metadata: Optional[Dict[str, str]] = None
) -> List[Document]:
    """Loads a PDF, filters out empty pages, and injects custom metadata."""
    pdf_path = Path(file_path)

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"Expected a PDF at: {pdf_path.resolve()}\n"
            "Please verify the file path."
        )

    loader = PyPDFLoader(str(pdf_path))
    valid_pages = []
    
    # Safely handle the case where no extra metadata is provided
    extra_metadata = extra_metadata or {}

    for page in loader.lazy_load():
        if not page.page_content.strip():
            continue  
            
        # 1. Update with a default source (always useful for tracking)
        page.metadata.update({"source": pdf_path.name})
        
        # 2. Inject whatever custom metadata the user passed in
        page.metadata.update(extra_metadata)
        
        valid_pages.append(page)

    if not valid_pages:
        raise ValueError(
            f"The PDF '{pdf_path.name}' loaded, but no extractable text was found. "
            "This usually means the PDF is scanned and needs OCR first."
        )

    logger.info(f"Loaded {len(valid_pages)} text-containing pages from {pdf_path.name}.")
    
    return valid_pages

In [9]:
pages = load_pdf_documents("Data/Big_Book_Of_Data_Engineering.pdf")
print(pages[12].page_content[:7500])
print("\nMetadata:", pages[0].metadata)

INFO:__main__:Loaded 127 text-containing pages from Big_Book_Of_Data_Engineering.pdf.


Conclusion  
As organizations strive to innovate with AI, data engineering is a focal point for 
success by delivering reliable, real-time data pipelines that make AI possible. 
With the Databricks Platform, built on a lakehouse architecture and powered by 
Data Intelligence, data engineers are set up for success in dealing with the critical 
challenges posed in the modern data landscape. By leaning on the advanced 
capabilities of the Data Intelligence Platform, data engineers don’t need to spend 
as much time managing complex pipelines or dealing with reliability, scalability 
and data quality issues. Instead, they can focus on innovation and bringing more 
value to the organization.
FOLLOW PROVEN BEST PRACTICES 
In the next section, we describe best practices for data engineering and  
end-to-end use cases drawn from real-world examples. From data ingestion  
and real-time processing to orchestration and data federation, you’ll learn how 
to apply pr
oven patterns and make the best 

### ✂️ 2. Recursive Chunking Dynamics

A full PDF page is too large and covers too many mixed topics for high-quality retrieval. We split it into smaller overlapping chunks.

**Why `RecursiveCharacterTextSplitter`?**
Unlike standard splitters that cut blindly at character limits, this splitter tries to preserve semantic meaning by splitting on natural boundaries in a prioritized order:
1. `\n\n` (Paragraph boundaries - keeps whole ideas together)
2. `\n` (Line breaks)
3. ` ` (Word boundaries)

**Why `add_start_index=True`?**
It saves the exact character offset in the source document. This gives your application precise traceability back to the original text location for citations.

We will start with chunks of **1,000 characters** and **200 characters of overlap**.

- **Chunk size**: controls how much text each vector represents.
- **Chunk overlap**: keeps nearby context from being lost at chunk boundaries.

In [10]:
def chunk_documents(
    documents: List[Document], 
    chunk_size: int = 1000, 
    chunk_overlap: int = 200
) -> List[Document]:
    """Splits a list of documents into chunks and prints a preview of the first one."""
    
    if not documents:
        logger.warning("No documents were provided to split. Returning an empty list.")
        return []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )

    splits = text_splitter.split_documents(documents)

    logger.info(
        f"Split {len(documents)} pages into {len(splits)} chunks "
        f"(Size: {chunk_size}, Overlap: {chunk_overlap})."
    )

    # --- Inspection logic added directly here ---
    if splits:
        sample_chunk = splits[0]
        print("\n--- Sample Chunk [0] Preview ---")
        print(sample_chunk.page_content[:750])
        print("\nMetadata:", sample_chunk.metadata)
        print("--------------------------------\n")

    return splits

In [11]:
chunks = chunk_documents(pages, chunk_size=1000, chunk_overlap=200)

INFO:__main__:Split 127 pages into 332 chunks (Size: 1000, Overlap: 200).



--- Sample Chunk [0] Preview ---
sales@3cloudsolutions.com // 888-88-AZURE // 3cloudsolutions.com

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.1 (Windows)', 'creationdate': '2025-01-24T01:32:23+00:00', 'moddate': '2025-10-08T18:49:08-05:00', 'trapped': '/False', 'source': 'Big_Book_Of_Data_Engineering.pdf', 'total_pages': 127, 'page': 0, 'page_label': '1', 'start_index': 0}
--------------------------------



### Chunk Size and Chunk Overlap

**Chunk size** sets how much text goes into each chunk. **Chunk overlap** sets how much text repeats between one chunk and the next, so context isn't lost at the cut-off point.

**Choosing chunk size**
- Too small → chunks miss context and feel fragmented.
- Too large → chunks mix multiple ideas, making retrieval less focused.

**Choosing chunk overlap**
- Too little → key details near chunk edges can get lost.
- Too much → chunks repeat the same content, wasting storage and slowing retrieval.

In short, it's a balance between **precision and context**: smaller chunks with less overlap retrieve more precisely but may lack surrounding detail, while larger chunks with more overlap preserve context but retrieve less precisely. The best settings depend on how dense and technical your source document is.

### 🧮 3. Embedding Mathematics & Qdrant Deployment

Before storing data, the "Librarian" embedding model converts text into numeric vectors. 

**How Similarity is Scored:**
We use **Cosine Similarity**, which measures the angle between two vectors rather than straight-line distance. The formula is:

$$ \text{cosine similarity}(a, b) = \frac{a \cdot b}{\|a\| \|b\|} $$

*   **1.0**: Pointing the exact same direction (highly similar).
*   **0.0**: Orthogonal (completely unrelated).

**Qdrant Deployment Options:**
In this notebook, we use `location=":memory:"`. Here is how you can transition this exact code to a production environment later:
*   **Prototype:** `location=":memory:"` (Lightning fast, but data deletes on restart).
*   **Local Dev:** `path="./qdrant_data"` (Saves to a local folder for persistence).
*   **Production:** `url="https://[cluster].qdrant.tech"`, `api_key="..."` (Fully managed cloud).

*Note: We wrap our embedding model in a `Tenacity` retry block. Free-tier API keys often throw `HTTP 429 Too Many Requests` during bulk embedding. This makes the pipeline resilient.*

In [12]:
class SafeGoogleEmbeddings(GoogleGenerativeAIEmbeddings):
    
    # Changed to wait exactly 60 seconds, and gave it a generous 10 attempts
    @retry(wait=wait_fixed(60), stop=stop_after_attempt(10))
    def embed_documents(self, texts, *args, **kwargs):
        print(f"Embedding batch of {len(texts)} chunks...")
        return super().embed_documents(texts, *args, **kwargs)

# 1. Initialize your new safe class
embedding_model = "models/gemini-embedding-001"
embeddings = SafeGoogleEmbeddings(model=embedding_model)

collection_name = "data_engineering_ebook"

# 2. Run the pipeline
print("Starting vector store creation (this may take a few minutes if rate limited)...")

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with Gemini embedding model")
print(f"Built in-memory Qdrant collection: {collection_name}")

Starting vector store creation (this may take a few minutes if rate limited)...
Embedding batch of 1 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 64 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 64 chunks...


...

Embedding batch of 64 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 12 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedded chunks with Gemini embedding model
Built in-memory Qdrant collection: data_engineering_ebook


### Checking Retrieval Quality

Before generating any answers, it's important to look at what the retriever actually returns. If the retrieved chunks are irrelevant, the generated answer will be irrelevant too, regardless of how good the model is.

The parameter `k` controls how many chunks come back for each query:
- A higher `k` gives more context, but increases the chance of noisy or unrelated chunks.
- A lower `k` is more focused, but might miss useful supporting details.

We'll start with `k = 4` and revisit this value later if retrieval quality needs improvement.

Each retrieved chunk comes with a similarity score, calculated the same way as in the embedding primer earlier, just applied to full chunks instead of single words.

In [19]:
def display_retrieval_results(
    vector_store: VectorStore, 
    query: str, 
    k: int = 3
) -> list[tuple[Document, float]]:
    """
    Runs a similarity search and prints a readable summary of the top matches.
    
    Args:
        vector_store: The LangChain vector store instance to search.
        query: The search query string.
        k: The number of top documents to retrieve.
        
    Returns:
        A list of tuples containing the Document and its similarity score.
    """
    results = vector_store.similarity_search_with_score(query, k=k)
    
    if not results:
        print(f"No results found for: '{query}'")
        return []

    print(f"\nTop {len(results)} matches for: '{query}'\n" + "="*80)

    for index, (doc, score) in enumerate(results, start=1):
        # 1. Safely extract metadata using a local reference
        meta = doc.metadata
        page = meta.get("page")
        page_display = page + 1 if isinstance(page, int) else "Unknown"
        start_idx = meta.get("start_index", "Unknown")
        
        # 2. Clean and smartly truncate the text
        clean_text = doc.page_content.replace("\n", " ").strip()
        preview = textwrap.shorten(clean_text, width=350, placeholder=" ...")

        # 3. Print formatted output
        print(f"Match {index} | Score: {score:.3f} | Page: {page_display} | Start: {start_idx}")
        print(preview)
        print("-" * 80)

    return results

In [20]:
matches = display_retrieval_results(vector_store, "What are the key components of a modern data engineering pipeline?", k=4)

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"



Top 4 matches for: 'What are the key components of a modern data engineering pipeline?'
Match 1 | Score: 0.742 | Page: 4 | Start: 1610
framework, which includes three main parts: 1. Ingest Data ing estion is the process of bringing data from one or more data sources into a data platform. These data sources can be files stored on-premises or on cloud storage services, databases, applications and, increasingly, data streams that produce real-time events. 2. Transform Data tr ...
--------------------------------------------------------------------------------
Match 2 | Score: 0.741 | Page: 5 | Start: 819
factory floors, more and more data is created and streamed in real time and requires low-latency processing so it can be used in real-time decision-making. ■ Scaling data pipelines reliably: With data coming in large quantities and often in real time, scaling the compute infrastructure that runs data pipelines is challenging, especially when ...
------------------------------------------

### 🔗 4. LCEL Pipeline and The Generation Model

Now we define the "Analyst" (the Generator) and chain everything together using **LCEL (LangChain Expression Language)**.

**What is the `|` operator doing?**
LCEL overloads Python's bitwise OR operator to act like a Unix pipe. It automatically passes the output of the left component as the input to the right component:
`Prompt Formatting -> API Call to Gemini -> Extracting Raw String`

**The Importance of Temperature in RAG:**
We set `temperature=0.1`. Temperature controls model randomness (0.0 to 1.0+). 
*   **High (0.8):** Creative, takes risks (good for poetry).
*   **Low (0.1):** Deterministic, strict, analytical.
In RAG, you *must* use a low temperature to prevent hallucinations. We want the model to strictly synthesize the provided context, not invent new facts.

This is where retrieval and generation come together. Instead of asking the model to answer purely from what it already "knows," we first fetch relevant chunks from Qdrant, then hand those chunks to the model as grounding material before it writes an answer.

The workflow is intentionally simple — a two-step pattern rather than a full agent:

Retrieve the most relevant chunks for the question.

Generate an answer using only that retrieved context.

In [21]:
chat_model = "gemini-2.5-flash"
llm = ChatGoogleGenerativeAI(model=chat_model, temperature=0.1)

# 2. Prompts tailored for Data Engineering
RAG_SYSTEM_PROMPT = """You are a senior data engineering assistant in a vector RAG application.

Use only the provided context (which may include pipeline architectures, SQL scripts, or ETL/ELT documentation) to answer the user's question.
If the context does not contain enough information to formulate a complete technical answer, say: "I don't have enough information in the provided documentation to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Ensure that any SQL, Python, PySpark, or configuration code provided in your answer is strictly derived from the context.
Do not invent database schemas, DAG dependencies, or pipeline features that are not explicitly mentioned.
Keep the answer concise, technical, and directly applicable."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

# 3. Define the LCEL Chain
rag_chain = rag_prompt | llm | StrOutputParser()

### 🤖 5. Executing the RAG Architecture

This function combines the retrieval and generation phases. Keeping them explicitly separate in code makes debugging much easier (you can print `context` to see if a wrong answer was caused by bad retrieval or a hallucinating model).

In [22]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        # Safely handle missing metadata (e.g., markdown files may not have 'page')
        page = doc.metadata.get("page")
        page_display = f", page {page + 1}" if isinstance(page, int) else ""
        source = doc.metadata.get("source", "unknown_source")

        formatted_chunks.append(
            f"[Source {index}] {source}{page_display}, similarity score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)

In [23]:
def answer_question(vector_store, question: str, k: int = 4) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    # Retrieve documents
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    
    # Generate answer using Gemini
    answer = rag_chain.invoke({"context": context, "question": question})

    # Extract metadata cleanly
    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": round(score, 4), # Rounded for cleaner output
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs, # Raw docs included for debugging
    }

In [24]:
answer_k = 4
test_question = "What are the retry policies and dependencies for the daily sales Airflow DAG?"

# Pass the vector_store explicitly to avoid global variable scoping issues
result = answer_question(
    vector_store=vector_store, 
    question=test_question, 
    k=answer_k
)

print("### Answer:")
print(result["answer"])
print("\n### Sources:")
for source in result["sources"]:
    print(f"- {source['source_label']}: {source['file']} (Score: {source['score']})")

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


### Answer:
I don't have enough information in the provided documentation to answer that. The context discusses retry policies and dependencies within "Workflows" (likely Databricks Workflows, given mentions of Databricks SQL Editor and DLT) [Source 1, Source 2, Source 4], but it does not mention "Airflow DAG" or "daily sales."

### Sources:
- Source 1: Big_Book_Of_Data_Engineering.pdf (Score: 0.6836)
- Source 2: Big_Book_Of_Data_Engineering.pdf (Score: 0.6695)
- Source 3: Big_Book_Of_Data_Engineering.pdf (Score: 0.6607)
- Source 4: Big_Book_Of_Data_Engineering.pdf (Score: 0.6593)


### 🧪 6. Batch Testing & Guardrails Check

Rather than testing one question in isolation, we'll run several questions through the pipeline together. This makes it easier to spot patterns — for example, whether certain topics consistently retrieve weak matches, or whether the assistant correctly refuses to answer when a question falls outside the document's scope.

In [25]:
# Define test questions, including an out-of-domain query to test guardrails
test_questions = [
    "What are the core stages of a data engineering pipeline?",
    "How does batch processing differ from stream processing?",
    "What role does a data lakehouse play in modern architectures?",
    "What is the best recipe for making lasagna?", 
]

# Run the batch using our previously defined `answer_question` function
for question in test_questions:
    print("=" * 80)
    print(f"❓ QUESTION: {question}")
    
    # Generate the result
    result = answer_question(vector_store, question, k=4)
    
    print(f"\n🤖 GENERATED ANSWER:\n{result['answer']}")
    
    print("\n📚 TOP RETRIEVED SOURCES:")
    for source in result["sources"]:
        # We print the score to see how "confident" the vector search was
        print(f"   - {source['source_label']}: {source['file']} (Similarity Score: {source['score']})")
    print("=" * 80 + "\n")

❓ QUESTION: What are the core stages of a data engineering pipeline?


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



🤖 GENERATED ANSWER:
The core stages of a data engineering pipeline are:

1.  **Ingest:** This stage involves bringing data from one or more data sources (files, databases, applications, data streams) into a data platform [Source 1, Source 4].
2.  **Transform:** In this stage, raw ingested data undergoes a series of steps to filter, standardize, clean, and aggregate it for usable storage. A common pattern for this is the medallion architecture, which includes Bronze, Silver, and Gold stages [Source 1].
3.  **Orchestrate:** This refers to the scheduling and monitoring of the data pipeline, controlling its various steps, and handling failures (e.g., retries) [Source 1].

📚 TOP RETRIEVED SOURCES:
   - Source 1: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.7467)
   - Source 2: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.7147)
   - Source 3: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.7064)
   - Source 4: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.701

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



🤖 GENERATED ANSWER:
Batch processing and stream processing differ primarily in their application and how they handle data over time:

*   **Batch Processing:**
    *   Used for considered, strategic reasoning and systematic review of aggregate data collected over a period of time [Source 1].
    *   Most efficient and cost-effective method for processing large, aggregate volumes of data [Source 1].
    *   Can be done offline, reducing operating costs and providing greater control over the end-to-end process [Source 1].
    *   Typically meets the requirements of back-office functions, such as reporting, risk management, anomaly detection, and surveillance [Source 1, Source 3].

*   **Stream Processing (Real-time Systems):**
    *   Helps Financial Services Institutions (FSIs) remain agile in addressing volatile market conditions at the edge [Source 1].
    *   Supports real-time processing and decisioning, driven by the desire for instant gratification and rapidly changing customer n

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



🤖 GENERATED ANSWER:
A data lakehouse plays a crucial role in modern architectures by providing a unified, decoupled platform that supports all analytical and scientific data workloads [Source 3].

Its key roles include:
*   **Decoupled Design and Integration:** It offers a decoupled design that allows easy integration with other platforms in the modern data ecosystem [Source 1, Source 3].
*   **Simplified Ingestion and Serving:** It simplifies the construction of ingestion and serving layers, operating irrespective of the data’s source, volume, velocity, and destination [Source 1].
*   **Reliability and Performance:** Built on Delta Lake, it brings openness, reliability, and lifecycle management to data lakes, solving challenges related to unstructured/structured data ingestion, data quality, data deletion for compliance, and data modification [Source 2]. It combines the high scalability of data lakes with the high reliability of data warehouses [Source 4].
*   **Addressing Complex En

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



🤖 GENERATED ANSWER:
I don't have enough information in the provided documentation to answer that.

📚 TOP RETRIEVED SOURCES:
   - Source 1: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.5517)
   - Source 2: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.5443)
   - Source 3: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.5385)
   - Source 4: Big_Book_Of_Data_Engineering.pdf (Similarity Score: 0.5362)



### Tuning Retrieval Quality

If retrieval keeps returning weak matches for a particular question, try adjusting one variable at a time:

- **Chunk size** — smaller or larger pieces of text per chunk.
- **Chunk overlap** — how much context repeats between chunks.
- **Retrieval `k`** — how many chunks come back per query.
- **Query wording** — rephrasing the question itself can sometimes matter as much as the retrieval settings.

### Suggested workflow
1. Pick one question that performs poorly.
2. Look at its retrieved chunks and scores.
3. Change exactly one setting above.
4. Rebuild the text splitter and vector store with the new setting.
5. Re-run the question and compare the new scores and chunks against the old ones.

Changing only one variable at a time is what makes the comparison meaningful — adjusting several settings together makes it hard to know which change actually caused the improvement.

**Golden Rule of Experimentation:** Change *exactly one variable at a time*.

In [26]:
test_question = "How does batch processing differ from stream processing?"

print("🔴 BEFORE CHANGE")
# Test current performance
result_before = answer_question(vector_store, test_question, k=4)
for source in result_before["sources"]:
    print(f"Score: {source['score']} | {source['source_label']}")

🔴 BEFORE CHANGE


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Score: 0.7612 | Source 1
Score: 0.7169 | Source 2
Score: 0.7144 | Source 3
Score: 0.7112 | Source 4


In [27]:
def rebuild_vector_store(raw_documents, embeddings_model, chunk_size: int, chunk_overlap: int):
    """Utility to safely rebuild the in-memory Qdrant vector store with new text splitting parameters."""
    print(f"🔄 Rebuilding index: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}...")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    
    # Split the raw documents
    splits = text_splitter.split_documents(raw_documents)
    
    # Recreate the Qdrant store
    new_vector_store = QdrantVectorStore.from_documents(
        documents=splits,
        embedding=embeddings_model,
        location=":memory:", # Qdrant syntax for local memory
        collection_name="data_eng_docs",
        force_recreate=True,
    )
    
    print(f"✅ Successfully indexed {len(splits)} new chunks.")
    return new_vector_store

# ---------------------------------------------------------
# Apply the change: Increase chunk size and overlap
# (Assuming 'pages' and 'embeddings' are defined earlier in your notebook)
# ---------------------------------------------------------

new_chunk_size = 1500
new_chunk_overlap = 300

vector_store = rebuild_vector_store(
    raw_documents=pages, 
    embeddings_model=embeddings, 
    chunk_size=new_chunk_size, 
    chunk_overlap=new_chunk_overlap
)

🔄 Rebuilding index: chunk_size=1500, chunk_overlap=300...
Embedding batch of 1 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 64 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 64 chunks...


...

Embedding batch of 42 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 429 Too Many Requests"


Embedding batch of 42 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


✅ Successfully indexed 234 new chunks.


In [28]:
print("🟢 AFTER CHANGE")
# Re-test the exact same question with the newly built vector store
result_after = answer_question(vector_store, test_question, k=4)
for source in result_after["sources"]:
    print(f"Score: {source['score']} | {source['source_label']}")

🟢 AFTER CHANGE


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Score: 0.7321 | Source 1
Score: 0.7202 | Source 2
Score: 0.7088 | Source 3
Score: 0.7031 | Source 4
